<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/day-16-rag-diagnostics/rag-diagnostics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.5 MB/s eta 0:00:00


In [4]:
import time
import json
import numpy as np
import faiss
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
chat_model = genai.GenerativeModel('gemini-3.6-flash')

# Reuse Day 15's knowledge base, index, and retrieval function
knowledge_base = [
    "Nimbus Robotics was founded in 2031 by engineer Priya Kalathil in Pune, India.",
    "Nimbus Robotics' flagship product is the Aster-7, a warehouse picking robot with a 99.2% accuracy rate.",
    "The Aster-7 uses a proprietary gripper called FlexGrip, which adjusts pressure using 12 micro-sensors per finger.",
    "Nimbus Robotics reported revenue of 340 million rupees in fiscal year 2033.",
    "Nimbus Robotics' main competitor is Solace Automation, founded a year earlier in 2030.",
    "The Aster-7's battery lasts 14 hours on a single charge and recharges fully in 40 minutes.",
    "Nimbus Robotics employs 212 people across three offices: Pune, Bengaluru, and Singapore.",
    "The company's CTO, Rohan Mehta, previously led robotics research at a university lab for eight years.",
    "Nimbus Robotics' next product, the Aster-8, is scheduled for release in early 2035.",
    "The Aster-7 has been deployed in over 60 warehouses across South and Southeast Asia."
]

def get_embeddings(texts, model="models/gemini-embedding-001"):
    embeddings = []
    for text in texts:
        result = genai.embed_content(model=model, content=text)
        embeddings.append(result['embedding'])
        time.sleep(1)
    return embeddings

corpus_embeddings = get_embeddings(knowledge_base)
corpus_vectors = np.array(corpus_embeddings, dtype=np.float32)
dimension = corpus_vectors.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(corpus_vectors)
print(f"FAISS index size: {index.ntotal} vectors")

FAISS index size: 10 vectors


In [5]:
def retrieve_chunks_with_scores(query, top_k=3):
    query_embedding = get_embeddings([query])[0]
    query_vector = np.array([query_embedding], dtype=np.float32)
    distances, indices = index.search(query_vector, top_k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({"chunk": knowledge_base[idx], "distance": float(dist)})
    return results

def generate_with_rag_logged(query, top_k=3):
    """RAG call that logs everything needed for diagnosis."""
    retrieved = retrieve_chunks_with_scores(query, top_k=top_k)
    context = "\n".join(f"- {r['chunk']}" for r in retrieved)

    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the context doesn't contain the answer, say "I don't have that information."

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:"""

    response = chat_model.generate_content(prompt)
    answer = response.text

    log_entry = {
        "query": query,
        "retrieved_chunks": retrieved,
        "generated_answer": answer
    }
    return log_entry

In [6]:
test_suite = [
    # 1. RETRIEVAL FAILURE — question the KB genuinely doesn't cover
    {"query": "What color is the Aster-7 robot?", "failure_mode": "retrieval_failure"},
    {"query": "Does Nimbus Robotics have an office in the United States?", "failure_mode": "retrieval_failure"},
    {"query": "What programming language does the Aster-7 run on?", "failure_mode": "retrieval_failure"},

    # 2. CONTEXT WINDOW OVERFLOW — force top_k very high to dilute signal
    {"query": "Tell me everything about Nimbus Robotics", "failure_mode": "context_overflow", "top_k": 10},
    {"query": "Summarize all of Nimbus Robotics' products, people, and finances", "failure_mode": "context_overflow", "top_k": 10},
    {"query": "Give me a full company profile of Nimbus Robotics", "failure_mode": "context_overflow", "top_k": 10},

    # 3. ANSWER-CONTEXT MISMATCH — correct chunk retrieved, but check if answer actually matches it
    {"query": "How much revenue did Nimbus Robotics report?", "failure_mode": "answer_context_mismatch"},
    {"query": "How many people does Nimbus Robotics employ?", "failure_mode": "answer_context_mismatch"},
    {"query": "How long does the Aster-7 battery last?", "failure_mode": "answer_context_mismatch"},

    # 4. VAGUE CONTEXT RETRIEVED — ambiguous/broad queries
    {"query": "Tell me about the competition", "failure_mode": "vague_context"},
    {"query": "What's new with the company?", "failure_mode": "vague_context"},
    {"query": "How is the technology?", "failure_mode": "vague_context"},

    # 5. CORRECT CHUNK RETRIEVED BUT WRONG ANSWER GENERATED — needs manual check against retrieved chunk
    {"query": "Who is the CTO of Nimbus Robotics and what did they do before?", "failure_mode": "correct_chunk_wrong_answer"},
    {"query": "When was Nimbus Robotics founded relative to Solace Automation?", "failure_mode": "correct_chunk_wrong_answer"},
    {"query": "What makes the Aster-7's gripper different from a standard gripper?", "failure_mode": "correct_chunk_wrong_answer"},
]

print(f"Test suite size: {len(test_suite)} queries across 5 failure modes")

Test suite size: 15 queries across 5 failure modes


In [7]:
all_results = []

for i, test in enumerate(test_suite):
    print(f"[{i+1}/15] Running: {test['query']}")
    top_k = test.get('top_k', 3)

    log_entry = generate_with_rag_logged(test['query'], top_k=top_k)
    log_entry['expected_failure_mode'] = test['failure_mode']
    all_results.append(log_entry)

    time.sleep(15)  # stay under free-tier rate limit

print("\nAll 15 queries completed.")

[1/15] Running: What color is the Aster-7 robot?
[2/15] Running: Does Nimbus Robotics have an office in the United States?
[3/15] Running: What programming language does the Aster-7 run on?
[4/15] Running: Tell me everything about Nimbus Robotics


ERROR:tornado.access:503 POST /v1beta/models/gemini-3.6-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 39019.03ms


[5/15] Running: Summarize all of Nimbus Robotics' products, people, and finances
[6/15] Running: Give me a full company profile of Nimbus Robotics


ERROR:tornado.access:503 POST /v1beta/models/gemini-3.6-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 67967.84ms


[7/15] Running: How much revenue did Nimbus Robotics report?
[8/15] Running: How many people does Nimbus Robotics employ?
[9/15] Running: How long does the Aster-7 battery last?
[10/15] Running: Tell me about the competition
[11/15] Running: What's new with the company?
[12/15] Running: How is the technology?
[13/15] Running: Who is the CTO of Nimbus Robotics and what did they do before?
[14/15] Running: When was Nimbus Robotics founded relative to Solace Automation?
[15/15] Running: What makes the Aster-7's gripper different from a standard gripper?

All 15 queries completed.


In [8]:
with open('rag_diagnostic_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("Saved to rag_diagnostic_results.json")

# Print a readable summary
for r in all_results:
    print("=" * 70)
    print(f"Query: {r['query']}")
    print(f"Expected failure mode: {r['expected_failure_mode']}")
    print(f"Answer: {r['generated_answer']}")
    print(f"Top retrieved chunk (dist={r['retrieved_chunks'][0]['distance']:.4f}): {r['retrieved_chunks'][0]['chunk']}")
    print()

Saved to rag_diagnostic_results.json
Query: What color is the Aster-7 robot?
Expected failure mode: retrieval_failure
Answer: I don't have that information.
Top retrieved chunk (dist=0.4324): The Aster-7's battery lasts 14 hours on a single charge and recharges fully in 40 minutes.

Query: Does Nimbus Robotics have an office in the United States?
Expected failure mode: retrieval_failure
Answer: No. According to the context, Nimbus Robotics only has three offices, which are located in Pune, Bengaluru, and Singapore.
Top retrieved chunk (dist=0.3657): Nimbus Robotics employs 212 people across three offices: Pune, Bengaluru, and Singapore.

Query: What programming language does the Aster-7 run on?
Expected failure mode: retrieval_failure
Answer: I don't have that information.
Top retrieved chunk (dist=0.4457): The Aster-7's battery lasts 14 hours on a single charge and recharges fully in 40 minutes.

Query: Tell me everything about Nimbus Robotics
Expected failure mode: context_overflow
A

In [9]:
# FIX 1: Tighten similarity threshold to reject weak retrieval matches
def generate_with_rag_v2(query, top_k=3, distance_threshold=0.8):
    """
    Improved version: rejects retrieval if the best match's distance
    exceeds a threshold, avoiding forced answers on genuinely irrelevant queries.
    """
    retrieved = retrieve_chunks_with_scores(query, top_k=top_k)

    if retrieved[0]['distance'] > distance_threshold:
        return {
            "query": query,
            "retrieved_chunks": retrieved,
            "generated_answer": "I don't have enough relevant information to answer this confidently."
        }

    context = "\n".join(f"- {r['chunk']}" for r in retrieved)

    # FIX 2: Tightened system prompt — explicitly discourage synthesizing beyond context
    prompt = f"""You are a precise assistant. Answer using ONLY the exact facts in the context below.
Do not infer, combine, or generalize beyond what is explicitly stated.
If any part of the question isn't directly answered by the context, say so explicitly
rather than guessing.

CONTEXT:
{context}

QUESTION:
{query}

ANSWER:"""

    response = chat_model.generate_content(prompt)
    return {"query": query, "retrieved_chunks": retrieved, "generated_answer": response.text}

# Re-test one of the vague-context failures with the fix
result = generate_with_rag_v2("Tell me about the competition")
print(f"Fixed answer: {result['generated_answer']}")
print(f"Top match distance: {result['retrieved_chunks'][0]['distance']:.4f}")

Fixed answer: I don't have enough relevant information to answer this confidently.
Top match distance: 0.8040


In [10]:
scorecard = [
    # (query_number, retrieval_quality_1to5, answer_quality_1to5)
    (1, 5, 5), (2, 5, 5), (3, 4, 4),
    (4, 2, 3), (5, 2, 3), (6, 2, 3),
    (7, 5, 5), (8, 5, 5), (9, 5, 5),
    (10, 2, 2), (11, 2, 3), (12, 1, 2),
    (13, 4, 4), (14, 4, 3), (15, 4, 4),
]

retrieval_scores = [r for _, r, _ in scorecard]
answer_scores = [a for _, _, a in scorecard]

print(f"Average retrieval quality: {sum(retrieval_scores)/len(retrieval_scores):.2f} / 5")
print(f"Average answer quality: {sum(answer_scores)/len(answer_scores):.2f} / 5")

Average retrieval quality: 3.47 / 5
Average answer quality: 3.73 / 5
